# Change in active wildfires across Australian states/territories during the last 5 days

## Preparing Notebook

In [ ]:
# import necessary libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import os

# set working directory
os.chdir("/Users/silviazemp/Desktop/Uni/FS26/Python/project")
print(os.getcwd())

/Users/silviazemp/Desktop/Uni/FS26/Python/project


## Accessing Wildfire Data via API

In [113]:
# 1.
# access api url

## satellite: MODIS NRT -> MODIS has a better distribution of acquisition times, leaving less gaps in the final map, and Near Real Time for analysing live data. 
### However, Modis has a worse spatial resolution with 1km instead of 375m like VIIRS, but that is a trade-off I can bear
## area: '112,-44,154,-9' = bounding box coordinates for Australia 
## day range: '5' = data of the last 5 days (FIRMS does not let you load more data with one API request)
## date: None = most recent available data, so today's data

MAP_KEY = '4899a992545cbeb46f9fd0b6a025ef17'
api_url ='https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_NRT/112,-44,154,-9/5' 

# 2.
# read in the data from URL

df_fires = pd.read_csv(api_url)


# 3.
# have a first glimpse at the data

display(df_fires.head(5))
display(df_fires.shape)
df_fires["acq_date"].unique()

# Check if a column has NaNs
print(df_fires["latitude"].hasnans)
print(df_fires["longitude"].hasnans)
print(df_fires["frp"].hasnans)
print(df_fires["acq_time"].hasnans)
print(df_fires["acq_date"].hasnans)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight
0,-19.09042,128.54053,318.84,4.68,1.98,2026-05-10,50,Terra,MODIS,40,6.1NRT,290.42,111.91,D
1,-18.88069,121.93324,319.51,1.47,1.20,2026-05-10,50,Terra,MODIS,77,6.1NRT,298.11,23.08,D
2,-18.87909,121.91957,312.06,1.47,1.20,2026-05-10,50,Terra,MODIS,67,6.1NRT,297.68,11.42,D
3,-18.81422,121.96227,312.88,1.47,1.20,2026-05-10,50,Terra,MODIS,23,6.1NRT,297.30,11.99,D
4,-18.53441,125.09199,309.29,2.50,1.52,2026-05-10,50,Terra,MODIS,61,6.1NRT,295.16,22.87,D


(2379, 14)

False
False
False
False
False


## Cleaning and Rearranging Data

### Adding Datetime Column with active time 

In [116]:
# 1. 
# combine the acq_date and acq_time column to one acq_datetime column and set it to an active time format with pandas function to_datetime

## acq_date is a string in the format YYYY-MM_DD, 
## while acq_time is an integer in Greenwich Mean Time (e.g. 603 meaning 6:03), 
## so it needs to be converted to string too (with astype(str)),
## fill it up to 4 numbers with zeros, so that all times have the same length (with str.zfill(4), e.g. 603 -> 0603)
## and save it as the format '%Y-%m-%d %H%M'

df_fires['acq_datetime'] = pd.to_datetime(df_fires['acq_date'] + ' ' + df_fires['acq_time'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
df_fires.head()

print (f'Australia GMT timezone datetime value range: {df_fires['acq_datetime'].min()} to {df_fires['acq_datetime'].max()}')

# 2.
# convert GMT into local time?

# 3.
# # Set the timestamp column as the index ?
###hourly_data = hourly_data.set_index("timestamp")

# Notice how 'timestamp' drops down a level to become the index!
###display(hourly_data.head(3))



Australia GMT timezone datetime value range: 2026-05-10 00:50:00 to 2026-05-14 07:14:00


### Converting raw coordinates into geometries

In [117]:
# the projection EPSG:9473 is used for Australia, as it is recommended for national mapping

# convert latitude, longitude values into point geometry and change CRS from 4326 to 9473

gdf_fires = gpd.GeoDataFrame(
    df_fires, geometry=gpd.points_from_xy(df_fires.longitude, df_fires.latitude), crs="EPSG:4326").to_crs(epsg=9473)
print(gdf_fires.crs)
gdf_fires.sample(10)

EPSG:9473


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,acq_datetime,geometry
438,-21.94713,132.40601,439.30,1.14,1.07,2026-05-11,657,Aqua,MODIS,100,6.1NRT,305.69,697.85,D,2026-05-11 06:57:00,POINT (41603.648 -2361131.947)
1299,-30.56551,148.42795,305.01,1.51,1.21,2026-05-12,1100,Terra,MODIS,60,6.1NRT,284.87,15.50,N,2026-05-12 11:00:00,POINT (1554903.311 -3426814.063)
113,-17.54492,128.91562,314.72,3.80,1.82,2026-05-10,618,Aqua,MODIS,46,6.1NRT,299.11,41.13,D,2026-05-10 06:18:00,POINT (-327834.653 -1875806.975)
2298,-14.57842,125.60635,338.42,1.04,1.02,2026-05-14,714,Aqua,MODIS,88,6.1NRT,303.33,32.26,D,2026-05-14 07:14:00,POINT (-695687.065 -1562781.425)
24,-27.70668,114.31213,305.54,1.04,1.02,2026-05-10,54,Terra,MODIS,62,6.1NRT,291.33,6.49,D,2026-05-10 00:54:00,POINT (-1717665.568 -3125194.161)
9,-17.47562,122.15723,322.93,1.46,1.19,2026-05-10,50,Terra,MODIS,0,6.1NRT,297.72,26.99,D,2026-05-10 00:50:00,POINT (-1045831.395 -1904496.639)
2255,-15.23073,129.96811,313.54,1.83,1.33,2026-05-14,714,Aqua,MODIS,33,6.1NRT,297.45,12.39,D,2026-05-14 07:14:00,POINT (-220033.484 -1618646.563)
2008,-15.47353,130.11450,321.86,1.08,1.04,2026-05-14,9,Terra,MODIS,73,6.1NRT,298.03,14.53,D,2026-05-14 00:09:00,POINT (-203788.6 -1645063.854)
1617,-14.95600,130.09167,324.87,1.42,1.18,2026-05-13,637,Aqua,MODIS,76,6.1NRT,299.69,23.17,D,2026-05-13 06:37:00,POINT (-207104.444 -1588299.71)
1304,-16.98673,144.88664,300.99,1.74,1.29,2026-05-12,1102,Terra,MODIS,28,6.1NRT,289.17,8.32,N,2026-05-12 11:02:00,POINT (1373723.884 -1879534.913)


## Adding a boundary GeoPackage file of States/Territories for spatial analysis

In [119]:
# 1. Load the GeoPackage of Australian states and territories and ensure CRS are matching
## Source of the GeoPackage: Australian Bureau of Statistics 
## https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs-edition-3/jul2021-jun2026/access-and-downloads/digital-boundary-files
gdf_states = gpd.read_file(
    "data/raw/ASGS_2021_Main_Structure_GDA2020.gpkg",
    layer="STE_2021_AUST_GDA2020"
).to_crs(epsg=9473)
 # check for valid geometries as sjoin was not working

# 2. Perform the spatial join
## the strict inner option is chosen, cause fires outside any Australian territories should be dropped (the bounding box includes some parts of Indonesia or Papua New Guinea)
## within is chosen as fires are point data and are either within or outside a polygon, and we only want the ones inside
gdf_joined = gpd.sjoin(gdf_fires, gdf_states, how="inner", predicate="within")

# 3. View the joined attribute table
display(gdf_joined.head(3))

# 4. Clean Data (omit columns not needed, as the attribute table is quite long now)
gdf_cleaned = gdf_joined.drop(columns=["brightness", "scan", "track", "confidence", "version", "bright_t31", "index_right", "CHANGE_FLAG_2021", "CHANGE_LABEL_2021", "AREA_ALBERS_SQKM", "ASGS_LOCI_URI_2021"])
display(gdf_cleaned.head(3))

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,geometry,index_right,STATE_CODE_2021,STATE_NAME_2021,CHANGE_FLAG_2021,CHANGE_LABEL_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021
0,-19.09042,128.54053,318.84,4.68,1.98,2026-05-10,50,Terra,MODIS,40,...,POINT (-363058.088 -2047959.561),4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5
1,-18.88069,121.93324,319.51,1.47,1.20,2026-05-10,50,Terra,MODIS,77,...,POINT (-1057335.637 -2061451.76),4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5
2,-18.87909,121.91957,312.06,1.47,1.20,2026-05-10,50,Terra,MODIS,67,...,POINT (-1058782.441 -2061388.04),4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5


,latitude,longitude,acq_date,acq_time,satellite,instrument,frp,daynight,acq_datetime,geometry,STATE_CODE_2021,STATE_NAME_2021,AUS_CODE_2021,AUS_NAME_2021
0,-19.09042,128.54053,2026-05-10,50,Terra,MODIS,111.91,D,2026-05-10 00:50:00,POINT (-363058.088 -2047959.561),5,Western Australia,AUS,Australia
1,-18.88069,121.93324,2026-05-10,50,Terra,MODIS,23.08,D,2026-05-10 00:50:00,POINT (-1057335.637 -2061451.76),5,Western Australia,AUS,Australia
2,-18.87909,121.91957,2026-05-10,50,Terra,MODIS,11.42,D,2026-05-10 00:50:00,POINT (-1058782.441 -2061388.04),5,Western Australia,AUS,Australia


## Spatial Analysis: Count fires per State/Territory

In [ ]:
fire_count = gdf_cleaned.groupby("STATE_NAME_2021").size()
display(fire_count)

STATE_NAME_2021
New South Wales        151
Northern Territory     766
Queensland             134
South Australia         16
Tasmania                11
Victoria                27
Western Australia     1176
dtype: int64

## Preparing the Data for a Heatmap

In [ ]:
# 1.
# group geometries by time (hourly resolution)
gdf_cleaned["time_bin"]= gdf_cleaned["acq_datetime"].dt.floor("h") #ist nur nötig bei hourly distribution, sonst identisch mit datetime column

data = []
time_index = []

for time, group in gdf_cleaned.groupby("time_bin"):
    
    heat_data = group[["latitude", "longitude", "frp"]].values.tolist() # add fire radiative power to show intensity of fires
    
    data.append(heat_data)
    time_index.append(str(time))

# check if it worked
gdf_cleaned.sample(5)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,index_right,STATE_CODE_2021,STATE_NAME_2021,CHANGE_FLAG_2021,CHANGE_LABEL_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021,time_bin
1137,-19.12370,128.60828,391.62,1.03,1.01,2026-05-09,11,Terra,MODIS,100,...,4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5,2026-05-09 00:00:00
1619,-20.15251,122.57591,312.09,1.70,1.28,2026-05-10,52,Terra,MODIS,66,...,4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5,2026-05-10 00:00:00
595,-19.69885,123.22784,312.79,1.18,1.08,2026-05-07,1242,Terra,MODIS,53,...,4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5,2026-05-07 12:00:00
1878,-28.29480,114.99119,350.63,1.05,1.02,2026-05-10,754,Aqua,MODIS,96,...,4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5,2026-05-10 07:00:00
1600,-19.19046,135.01022,303.83,2.45,1.51,2026-05-09,2312,Terra,MODIS,40,...,6,7,Northern Territory,0,No change,AUS,Australia,1.348134e+06,http://linked.data.gov.au/dataset/asgsed3/STE/7,2026-05-09 23:00:00


## Visualising the data with a folium map with State/Territory Polygons and an animated Heatmap of the Fire Distribution

In [ ]:
import folium
from folium.plugins import HeatMapWithTime
import branca.colormap as cm

# 1. Create basemap for the extent of Australia
aus_map = folium.Map(
    location=[-25.5649, 133.1234], # use the coordinates of Australia's centre (25°56′49.3″S, 133°12′34.7″E) for the location
    zoom_start=4,
    tiles="CartoDB DarkMatter",  # a dark basemap to make heatmap stand out
)

# 2. Add State/Territory Polygons
folium.GeoJson(
    gdf_territories,
    tooltip=folium.GeoJsonTooltip(
        fields=["STATE_NAME_2021"],
        aliases=["State/Territory:"]
    ),
    style_function=lambda feature:{
        "fillColor": "dark grey", # somehow dark grey does not work
        "color": "white",
        "weight": 0.5,
    }
).add_to(aus_map)

# 3. Create heatmap with Fires GDF that shows intensity of different fires (frp)
HeatMapWithTime(
    data,
    index=time_index,
    radius=8,
    auto_play=True,
    max_opacity=0.8,
    gradient={  
        0.2: "blue",
        0.4: "lime",
        0.6: "yellow",
        0.8: "orange",
        1.0: "red"
    }
).add_to(aus_map)

# 4. Add legend to heatmap with branca colormap
vmin = gdf_fires["frp"].min()
vmax = gdf_fires["frp"].max()

colormap = cm.LinearColormap(
    colors=["blue", "yellow", "red"],
    vmin=vmin,
    vmax=vmax,
    caption="Fire Radiative Power (FRP) in Megawatts"
)

colormap.add_to(aus_map)

# save the map (display does not work bc data file is too big)
aus_map.save("animated_heatmap_MODIS.html")